# Titanic Dataset: Data Cleaning and Exploratory Data Analysis

## Week 2 Assignment

**Dataset:** Kaggle Titanic Competition  
**Input directory:** `/kaggle/input/competitions/titanic`

### Objective

The objective of this notebook is to:

- Load and inspect the Titanic dataset.
- Identify and handle missing values and duplicate records.
- Create useful new features from the existing columns.
- Perform univariate, bivariate, and multivariate exploratory data analysis.
- Study relationships between passenger characteristics and survival.
- Identify important patterns and trends.
- Export the cleaned dataset, summary tables, and visualizations.

The analysis primarily uses `train.csv` because it contains the `Survived` target column.

In [ ]:
# Import required libraries

import os
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

print("Libraries imported successfully.")

In [ ]:
# Define dataset and output paths

INPUT_DIR = "/kaggle/input/competitions/titanic"
TRAIN_PATH = os.path.join(INPUT_DIR, "train.csv")
TEST_PATH = os.path.join(INPUT_DIR, "test.csv")
SUBMISSION_PATH = os.path.join(INPUT_DIR, "gender_submission.csv")

OUTPUT_DIR = "/kaggle/working/titanic_week2_eda_outputs"
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")

os.makedirs(FIGURE_DIR, exist_ok=True)

required_files = [TRAIN_PATH, TEST_PATH, SUBMISSION_PATH]
missing_files = [path for path in required_files if not os.path.exists(path)]

if missing_files:
    raise FileNotFoundError(
        "The following Titanic files were not found:\n"
        + "\n".join(missing_files)
        + "\n\nMake sure the Titanic competition dataset is attached to the notebook."
    )

print("Dataset directory verified:", INPUT_DIR)
print("Output directory created:", OUTPUT_DIR)

In [ ]:
# Load Titanic files

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
gender_submission_df = pd.read_csv(SUBMISSION_PATH)

print("Training dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)
print("Sample submission shape:", gender_submission_df.shape)

print("\nFirst five rows of the training dataset:")
display(train_df.head())

## 1. Initial Data Inspection

This section examines:

- Dataset dimensions
- Column names and data types
- Descriptive statistics
- Missing values
- Duplicate rows
- Unique values in categorical columns

In [ ]:
# Display basic dataset information

print("=" * 70)
print("COLUMN NAMES")
print("=" * 70)
print(train_df.columns.tolist())

print("\n" + "=" * 70)
print("DATA TYPES AND NON-NULL COUNTS")
print("=" * 70)
train_df.info()

print("\n" + "=" * 70)
print("NUMERICAL SUMMARY")
print("=" * 70)
display(train_df.describe().T)

print("\n" + "=" * 70)
print("CATEGORICAL SUMMARY")
print("=" * 70)
display(train_df.describe(include=["object"]).T)

In [ ]:
# Examine missing values and duplicates

missing_summary_before = pd.DataFrame({
    "Missing Values": train_df.isna().sum(),
    "Missing Percentage": (train_df.isna().mean() * 100).round(2)
}).sort_values("Missing Percentage", ascending=False)

print("Missing-value summary before cleaning:")
display(missing_summary_before[missing_summary_before["Missing Values"] > 0])

duplicate_count = train_df.duplicated().sum()
duplicate_passenger_ids = train_df["PassengerId"].duplicated().sum()

print("Duplicate rows:", duplicate_count)
print("Duplicate PassengerId values:", duplicate_passenger_ids)

In [ ]:
# Inspect categorical values

categorical_columns = ["Survived", "Pclass", "Sex", "Embarked"]

for column in categorical_columns:
    print(f"\nValue counts for {column}:")
    display(
        train_df[column]
        .value_counts(dropna=False)
        .rename("Count")
        .to_frame()
    )

## 2. Data Cleaning and Feature Engineering

The following cleaning operations are performed:

1. Remove duplicate rows.
2. Fill missing `Age` values using the median age within each `Sex` and `Pclass` group.
3. Fill missing `Embarked` values using the mode.
4. Fill missing `Fare` values using the median as a safety measure.
5. Convert missing cabin values to `Unknown`.
6. Create `HasCabin` and `Deck` features.
7. Extract passenger title from `Name`.
8. Create `FamilySize` and `IsAlone`.
9. Create `AgeGroup` and `FareGroup`.

In [ ]:
# Create a copy for cleaning

cleaned_df = train_df.copy()
rows_before = len(cleaned_df)

# 1. Remove exact duplicate rows
cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)

# 2. Fill missing Age values using grouped median
grouped_age_median = cleaned_df.groupby(["Sex", "Pclass"])["Age"].transform("median")
cleaned_df["Age"] = cleaned_df["Age"].fillna(grouped_age_median)
cleaned_df["Age"] = cleaned_df["Age"].fillna(cleaned_df["Age"].median())

# 3. Fill missing Embarked values using mode
embarked_mode = cleaned_df["Embarked"].mode(dropna=True)[0]
cleaned_df["Embarked"] = cleaned_df["Embarked"].fillna(embarked_mode)

# 4. Fill missing Fare values using median
cleaned_df["Fare"] = cleaned_df["Fare"].fillna(cleaned_df["Fare"].median())

# 5. Create cabin-related features
cleaned_df["HasCabin"] = cleaned_df["Cabin"].notna().astype(int)
cleaned_df["Deck"] = cleaned_df["Cabin"].str[0].fillna("Unknown")
cleaned_df["Cabin"] = cleaned_df["Cabin"].fillna("Unknown")

# 6. Extract title from passenger name
cleaned_df["Title"] = (
    cleaned_df["Name"]
    .str.extract(r",\s*([^.]*)\.", expand=False)
    .str.strip()
)

common_titles = ["Mr", "Miss", "Mrs", "Master"]
cleaned_df["Title"] = cleaned_df["Title"].where(
    cleaned_df["Title"].isin(common_titles),
    "Other"
)

# 7. Create family-related features
cleaned_df["FamilySize"] = cleaned_df["SibSp"] + cleaned_df["Parch"] + 1
cleaned_df["IsAlone"] = (cleaned_df["FamilySize"] == 1).astype(int)

# 8. Create age groups
cleaned_df["AgeGroup"] = pd.cut(
    cleaned_df["Age"],
    bins=[0, 12, 18, 35, 60, np.inf],
    labels=["Child", "Teenager", "Young Adult", "Adult", "Senior"],
    include_lowest=True
)

# 9. Create fare groups using quartiles
cleaned_df["FareGroup"] = pd.qcut(
    cleaned_df["Fare"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

rows_after = len(cleaned_df)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Duplicate rows removed:", rows_before - rows_after)
print("Total missing values after cleaning:", int(cleaned_df.isna().sum().sum()))

display(cleaned_df.head())

In [ ]:
# Verify the cleaning results

missing_summary_after = pd.DataFrame({
    "Missing Values": cleaned_df.isna().sum(),
    "Missing Percentage": (cleaned_df.isna().mean() * 100).round(2)
}).sort_values("Missing Percentage", ascending=False)

print("Missing-value summary after cleaning:")
display(missing_summary_after.head(10))

cleaning_summary = pd.DataFrame({
    "Column or Issue": [
        "Duplicate rows",
        "Age",
        "Embarked",
        "Fare",
        "Cabin"
    ],
    "Treatment": [
        "Removed exact duplicate rows",
        "Filled with median within Sex and Pclass groups",
        "Filled with the most frequent value",
        "Filled with overall median",
        "Missing values replaced by Unknown; HasCabin and Deck created"
    ]
})

print("\nCleaning operations:")
display(cleaning_summary)

print("\nCleaned dataset shape:", cleaned_df.shape)
print("Cleaned dataset columns:")
print(cleaned_df.columns.tolist())

## 3. Exploratory Data Analysis

The EDA investigates:

- Overall survival distribution
- Survival by gender
- Survival by passenger class
- Combined effect of gender and passenger class
- Age patterns
- Fare patterns
- Family size and solo travel
- Embarkation port
- Cabin availability
- Passenger title
- Correlations among numerical variables

In [ ]:
# Helper function for saving figures

def save_figure(filename):
    path = os.path.join(FIGURE_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

In [ ]:
# Overall survival distribution

survival_count_table = (
    cleaned_df["Survived"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Did Not Survive", 1: "Survived"})
    .to_frame("Passenger Count")
)

survival_percentage_table = (
    cleaned_df["Survived"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .rename(index={0: "Did Not Survive", 1: "Survived"})
    .to_frame("Percentage")
)

print("Survival counts:")
display(survival_count_table)

print("Survival percentages:")
display(survival_percentage_table)

plot_df = cleaned_df.copy()
plot_df["Survival Status"] = plot_df["Survived"].map({
    0: "Did Not Survive",
    1: "Survived"
})

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=plot_df, x="Survival Status")
ax.set_title("Distribution of Passenger Survival")
ax.set_xlabel("Survival Status")
ax.set_ylabel("Number of Passengers")

for container in ax.containers:
    ax.bar_label(container)

save_figure("01_survival_distribution.png")

In [ ]:
# Survival rate by gender

gender_survival = (
    cleaned_df.groupby("Sex")["Survived"]
    .agg(Total_Passengers="count", Survivors="sum", Survival_Rate="mean")
)

gender_survival["Survival_Rate_Percentage"] = (
    gender_survival["Survival_Rate"] * 100
).round(2)

display(gender_survival)

plt.figure(figsize=(7, 5))
ax = sns.barplot(
    data=cleaned_df,
    x="Sex",
    y="Survived",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Gender")
ax.set_xlabel("Gender")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f")

save_figure("02_survival_by_gender.png")

In [ ]:
# Survival rate by passenger class

class_survival = (
    cleaned_df.groupby("Pclass")["Survived"]
    .agg(Total_Passengers="count", Survivors="sum", Survival_Rate="mean")
)

class_survival["Survival_Rate_Percentage"] = (
    class_survival["Survival_Rate"] * 100
).round(2)

display(class_survival)

plt.figure(figsize=(7, 5))
ax = sns.barplot(
    data=cleaned_df,
    x="Pclass",
    y="Survived",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Passenger Class")
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f")

save_figure("03_survival_by_class.png")

In [ ]:
# Combined relationship between gender, passenger class, and survival

gender_class_table = pd.crosstab(
    index=cleaned_df["Pclass"],
    columns=cleaned_df["Sex"],
    values=cleaned_df["Survived"],
    aggfunc="mean"
).mul(100).round(2)

print("Survival rate (%) by passenger class and gender:")
display(gender_class_table)

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=cleaned_df,
    x="Pclass",
    y="Survived",
    hue="Sex",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Gender and Passenger Class")
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

save_figure("04_survival_by_gender_and_class.png")

In [ ]:
# Age distribution and survival

plt.figure(figsize=(10, 5))
sns.histplot(
    data=cleaned_df,
    x="Age",
    hue="Survived",
    bins=30,
    kde=True,
    element="step"
)
plt.title("Age Distribution by Survival Status")
plt.xlabel("Age")
plt.ylabel("Passenger Count")
save_figure("05_age_distribution_by_survival.png")

age_group_survival = (
    cleaned_df.groupby("AgeGroup", observed=False)["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
)
age_group_survival["Survival_Rate_Percentage"] = (
    age_group_survival["Survival_Rate"] * 100
).round(2)

display(age_group_survival)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=cleaned_df,
    x="AgeGroup",
    y="Survived",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Age Group")
ax.set_xlabel("Age Group")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)
plt.xticks(rotation=15)

save_figure("06_survival_by_age_group.png")

In [ ]:
# Fare distribution and survival

fare_summary = (
    cleaned_df.groupby("Survived")["Fare"]
    .agg(Passengers="count", Mean_Fare="mean", Median_Fare="median", Maximum_Fare="max")
    .rename(index={0: "Did Not Survive", 1: "Survived"})
    .round(2)
)

display(fare_summary)

plt.figure(figsize=(8, 5))
sns.boxplot(
    data=cleaned_df,
    x="Survived",
    y="Fare",
    showfliers=False
)
plt.title("Fare Distribution by Survival Status")
plt.xlabel("Survival Status")
plt.ylabel("Fare")
plt.xticks([0, 1], ["Did Not Survive", "Survived"])
save_figure("07_fare_by_survival.png")

fare_group_survival = (
    cleaned_df.groupby("FareGroup", observed=False)["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
)
fare_group_survival["Survival_Rate_Percentage"] = (
    fare_group_survival["Survival_Rate"] * 100
).round(2)

display(fare_group_survival)

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=cleaned_df,
    x="FareGroup",
    y="Survived",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Fare Group")
ax.set_xlabel("Fare Group")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

save_figure("08_survival_by_fare_group.png")

In [ ]:
# Family size and travelling alone

family_size_survival = (
    cleaned_df.groupby("FamilySize")["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
)
family_size_survival["Survival_Rate_Percentage"] = (
    family_size_survival["Survival_Rate"] * 100
).round(2)

display(family_size_survival)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=cleaned_df,
    x="FamilySize",
    y="Survived",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Family Size")
ax.set_xlabel("Family Size")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

save_figure("09_survival_by_family_size.png")

alone_table = (
    cleaned_df.groupby("IsAlone")["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
    .rename(index={0: "With Family", 1: "Alone"})
)
alone_table["Survival_Rate_Percentage"] = (
    alone_table["Survival_Rate"] * 100
).round(2)

display(alone_table)

alone_plot_df = cleaned_df.copy()
alone_plot_df["Travel Status"] = alone_plot_df["IsAlone"].map({
    0: "With Family",
    1: "Alone"
})

plt.figure(figsize=(7, 5))
ax = sns.barplot(
    data=alone_plot_df,
    x="Travel Status",
    y="Survived",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate: Alone vs With Family")
ax.set_xlabel("Travel Status")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

save_figure("10_survival_alone_vs_family.png")

In [ ]:
# Embarkation port and survival

embarked_names = {
    "C": "Cherbourg",
    "Q": "Queenstown",
    "S": "Southampton"
}

embarked_table = (
    cleaned_df.groupby("Embarked")["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
)
embarked_table.index = embarked_table.index.map(embarked_names)
embarked_table["Survival_Rate_Percentage"] = (
    embarked_table["Survival_Rate"] * 100
).round(2)

display(embarked_table)

embarked_plot_df = cleaned_df.copy()
embarked_plot_df["Embarkation Port"] = embarked_plot_df["Embarked"].map(embarked_names)

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=embarked_plot_df,
    x="Embarkation Port",
    y="Survived",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Embarkation Port")
ax.set_xlabel("Embarkation Port")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

save_figure("11_survival_by_embarkation_port.png")

In [ ]:
# Cabin availability and survival

cabin_table = (
    cleaned_df.groupby("HasCabin")["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
    .rename(index={0: "Cabin Not Recorded", 1: "Cabin Recorded"})
)
cabin_table["Survival_Rate_Percentage"] = (
    cabin_table["Survival_Rate"] * 100
).round(2)

display(cabin_table)

cabin_plot_df = cleaned_df.copy()
cabin_plot_df["Cabin Status"] = cabin_plot_df["HasCabin"].map({
    0: "Cabin Not Recorded",
    1: "Cabin Recorded"
})

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=cabin_plot_df,
    x="Cabin Status",
    y="Survived",
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Cabin Availability")
ax.set_xlabel("Cabin Information")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

save_figure("12_survival_by_cabin_availability.png")

In [ ]:
# Passenger title and survival

title_survival = (
    cleaned_df.groupby("Title")["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
    .sort_values("Survival_Rate", ascending=False)
)
title_survival["Survival_Rate_Percentage"] = (
    title_survival["Survival_Rate"] * 100
).round(2)

display(title_survival)

plt.figure(figsize=(8, 5))
title_order = title_survival.index.tolist()
ax = sns.barplot(
    data=cleaned_df,
    x="Title",
    y="Survived",
    order=title_order,
    estimator=np.mean,
    errorbar=None
)
ax.set_title("Survival Rate by Passenger Title")
ax.set_xlabel("Passenger Title")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)

save_figure("13_survival_by_title.png")

In [ ]:
# Correlation analysis

numeric_columns = [
    "Survived",
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone",
    "HasCabin"
]

correlation_matrix = cleaned_df[numeric_columns].corr()

display(correlation_matrix.round(3))

plt.figure(figsize=(10, 7))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True
)
plt.title("Correlation Heatmap of Numerical Variables")

save_figure("14_correlation_heatmap.png")

## 4. Key Findings

The following cell calculates the main findings directly from the cleaned dataset.  
This ensures that the written findings match the executed results.

In [ ]:
# Calculate major findings

overall_survival = cleaned_df["Survived"].mean() * 100

female_survival = cleaned_df.loc[
    cleaned_df["Sex"] == "female", "Survived"
].mean() * 100

male_survival = cleaned_df.loc[
    cleaned_df["Sex"] == "male", "Survived"
].mean() * 100

first_class_survival = cleaned_df.loc[
    cleaned_df["Pclass"] == 1, "Survived"
].mean() * 100

third_class_survival = cleaned_df.loc[
    cleaned_df["Pclass"] == 3, "Survived"
].mean() * 100

child_survival = cleaned_df.loc[
    cleaned_df["AgeGroup"] == "Child", "Survived"
].mean() * 100

alone_survival_rate = cleaned_df.loc[
    cleaned_df["IsAlone"] == 1, "Survived"
].mean() * 100

with_family_survival_rate = cleaned_df.loc[
    cleaned_df["IsAlone"] == 0, "Survived"
].mean() * 100

survivor_median_fare = cleaned_df.loc[
    cleaned_df["Survived"] == 1, "Fare"
].median()

non_survivor_median_fare = cleaned_df.loc[
    cleaned_df["Survived"] == 0, "Fare"
].median()

cabin_recorded_survival = cleaned_df.loc[
    cleaned_df["HasCabin"] == 1, "Survived"
].mean() * 100

cabin_unknown_survival = cleaned_df.loc[
    cleaned_df["HasCabin"] == 0, "Survived"
].mean() * 100

findings = [
    f"The overall passenger survival rate was {overall_survival:.2f}%.",
    (
        f"Female passengers had a survival rate of {female_survival:.2f}%, "
        f"compared with {male_survival:.2f}% for male passengers."
    ),
    (
        f"First-class passengers had a survival rate of {first_class_survival:.2f}%, "
        f"while third-class passengers had a survival rate of {third_class_survival:.2f}%."
    ),
    f"Children had a survival rate of approximately {child_survival:.2f}%.",
    (
        f"Passengers travelling alone had a survival rate of {alone_survival_rate:.2f}%, "
        f"compared with {with_family_survival_rate:.2f}% for passengers travelling with family."
    ),
    (
        f"The median fare was {survivor_median_fare:.2f} for survivors and "
        f"{non_survivor_median_fare:.2f} for non-survivors."
    ),
    (
        f"Passengers with recorded cabin information had a survival rate of "
        f"{cabin_recorded_survival:.2f}%, compared with {cabin_unknown_survival:.2f}% "
        f"for passengers without recorded cabin information."
    ),
    (
        "Gender and passenger class showed some of the clearest relationships with survival. "
        "Fare, family size, age group, title, embarkation port, and cabin availability also "
        "showed noticeable patterns."
    ),
    (
        "These relationships show association rather than direct causation. For example, "
        "fare and cabin information are also strongly connected to passenger class."
    )
]

print("=" * 80)
print("MAJOR EDA FINDINGS")
print("=" * 80)

for number, finding in enumerate(findings, start=1):
    print(f"{number}. {finding}")

## 5. Conclusion

The Titanic dataset was successfully cleaned and analysed. Missing age values were filled using grouped medians based on gender and passenger class, missing embarkation values were filled using the mode, and missing cabin information was represented using new cabin-related features.

The exploratory analysis indicates that survival was not randomly distributed among passengers. Gender and passenger class had especially strong relationships with survival. Female passengers and first-class passengers were much more likely to survive than male passengers and third-class passengers.

Age, fare, family size, travelling-alone status, title, embarkation port, and cabin information also displayed meaningful patterns. Children generally showed favourable survival outcomes, and passengers travelling with family often performed better than those travelling alone. Higher fares and recorded cabin information were also associated with higher survival, although these factors were related to passenger class.

Overall, the analysis demonstrates how data cleaning, feature engineering, statistical summaries, and visualizations can be used to discover patterns and relationships in a real-world dataset.

In [ ]:
# Export cleaned data, tables, findings, and figures

cleaned_csv_path = os.path.join(OUTPUT_DIR, "titanic_cleaned.csv")
summary_csv_path = os.path.join(OUTPUT_DIR, "eda_summary.csv")
missing_before_path = os.path.join(OUTPUT_DIR, "missing_values_before_cleaning.csv")
missing_after_path = os.path.join(OUTPUT_DIR, "missing_values_after_cleaning.csv")
findings_path = os.path.join(OUTPUT_DIR, "key_findings.txt")

cleaned_df.to_csv(cleaned_csv_path, index=False)
missing_summary_before.to_csv(missing_before_path)
missing_summary_after.to_csv(missing_after_path)

eda_summary = pd.DataFrame({
    "Metric": [
        "Original rows",
        "Cleaned rows",
        "Original columns",
        "Cleaned columns",
        "Duplicate rows removed",
        "Overall survival rate (%)",
        "Female survival rate (%)",
        "Male survival rate (%)",
        "First-class survival rate (%)",
        "Third-class survival rate (%)",
        "Children survival rate (%)",
        "Travelling alone survival rate (%)",
        "Travelling with family survival rate (%)"
    ],
    "Value": [
        len(train_df),
        len(cleaned_df),
        train_df.shape[1],
        cleaned_df.shape[1],
        len(train_df) - len(cleaned_df),
        round(overall_survival, 2),
        round(female_survival, 2),
        round(male_survival, 2),
        round(first_class_survival, 2),
        round(third_class_survival, 2),
        round(child_survival, 2),
        round(alone_survival_rate, 2),
        round(with_family_survival_rate, 2)
    ]
})

eda_summary.to_csv(summary_csv_path, index=False)

with open(findings_path, "w", encoding="utf-8") as file:
    file.write("Titanic Data Cleaning and EDA - Key Findings\n")
    file.write("=" * 55 + "\n\n")
    for number, finding in enumerate(findings, start=1):
        file.write(f"{number}. {finding}\n")

archive_base = "/kaggle/working/titanic_week2_eda_outputs"
zip_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=OUTPUT_DIR
)

print("Export completed successfully.")
print("\nGenerated files:")
print("-", cleaned_csv_path)
print("-", summary_csv_path)
print("-", missing_before_path)
print("-", missing_after_path)
print("-", findings_path)
print("-", FIGURE_DIR)
print("-", zip_path)

print("\nEDA summary:")
display(eda_summary)

## Submission Checklist

After clicking **Save Version → Save & Run All**, submit:

1. This completed Kaggle notebook.
2. `titanic_cleaned.csv`
3. `eda_summary.csv`
4. `key_findings.txt`
5. The generated figures, if required.
6. `titanic_week2_eda_outputs.zip`, which contains the complete exported output folder.

All generated files will be available in the Kaggle notebook **Output** section.